In [ ]:
#| default_exp execute

## Executing notebooks

Execute notebooks with visible outputs, local package imports, and a fast per-cell timeout. Cells that exceed the timeout are marked with source-hash metadata and skipped on later runs until their source changes.

Execution closes the loop after reading and writing. The project needs a way to run notebooks as notebooks, with local imports available and with visible outputs copied back into the notebook for inspection.

This notebook wraps `execnb` with nbdev-friendly defaults: local import paths, optional partial execution, timeout handling, and a test helper that reports notebook errors in a concise form.

Execution is the confidence step after an edit. The wrapper keeps imports close to how nbdev users run notebooks, records visible outputs for inspection, and marks timed-out cells so repeated runs do not keep blocking on the same long operation.

```python
exec_nb("nbs/03_execute.ipynb", up2id="some-cell-id", timeout=10, show_output=True)
```

In [0]:
import tempfile as _tempfile
from pathlib import Path as _Path
from nbskill.execute import exec_nb
from nbskill.write import write_nb

with _tempfile.TemporaryDirectory() as td:
    path = _Path(td) / "demo.ipynb"
    write_nb(str(path), "%%code\nvalue = 6 * 7\nprint(value)", replace=True, export=False)
    exec_nb(str(path), timeout=5, show_output=True)

nbskill: cell id=762cae87 ran longer than 20s and was stopped.


Wrote 1 cells to /var/folders/6_/45pyyxdd7hz3wz33p813bx_c0000gn/T/tmpx9q6inow/demo.ipynb using replace
---------------------------------------------------------------------------
TimeoutError                              Traceback (most recent call last)
Cell In[1], line 9
      7 path = _Path(td) / "demo.ipynb"
      8 write_nb(str(path), "%%code\nvalue = 6 * 7\nprint(value)", replace=True, export=False)
----> 9 exec_nb(str(path), timeout=5, show_output=True)

File ~/.local/share/uv/tools/nbskill/lib/python3.13/site-packages/fastcore/script.py:158, in call_parse.<locals>._f(*args, **kwargs)
    156 @wraps(func)
    157 def _f(*args, **kwargs):
--> 158     if args or kwargs or _in_call_parse.get(): return func(*args, **kwargs)
    159     with set_ctx(_in_call_parse):
    160         mod = inspect.getmodule(inspect.currentframe().f_back)

File ~/Projects/nbskill/nbskill/foundation.py:153, in tracked_call.<locals>.wrapper(*args, **kwargs)
    150 @wraps(func)
    151 def wrapper(*args, 

TimeoutError: 

In [ ]:
#| export
import hashlib
import sys
from pathlib import Path

from execnb.shell import CaptureShell
from fastcore.nbio import read_nb as _read_nb
from fastcore.nbio import write_nb as _write_nb
from fastcore.script import call_parse

from nbskill.foundation import (
    cell_metadata, cli_error, cli_return, one_chapter, parse_literal,
    stamp_notebook_metadata, tracked_call,
)
from nbskill.parallel import execution_slot, notebook_locks

### Choosing how much to run

Sometimes verification only needs the first few cells or a single chapter. These helpers translate an index, a cell id, or a chapter into pre/post hooks that stop execution at the right point.

In [ ]:
#| export
def _exec_limiters(up2id):
    up2id = parse_literal(up2id)
    noop = lambda cell: None
    if up2id is None: return (lambda cell: False), noop
    if isinstance(up2id, int):
        if up2id < 0: raise ValueError("up2id must be >= 0")
        return (lambda cell: cell.idx_ >= up2id), noop

    done = False
    def preproc(cell): return done
    def postproc(cell):
        nonlocal done
        if cell.id == str(up2id): done = True
    return preproc, postproc

### Importing like the project does

Notebook execution should see the same local package that tests and examples see. This section finds the project root from common markers and puts the notebook folder, root, and optional `src` folder on the shell path.

In [ ]:
#| export
def _project_root_for_notebook(path):
    path = Path(path).resolve()
    start = path.parent if path.suffix else path
    markers = ("pyproject.toml", "settings.ini", "nbdev.yml", ".git")
    for folder in (start, *start.parents):
        if any((folder / marker).exists() for marker in markers): return folder
    if start.name in {"nbs", "notebooks"} and start.parent != start.parent.parent: return start.parent
    return start

In [ ]:
#| export
def _local_import_paths(path):
    nb_dir = Path(path).resolve().parent
    root = _project_root_for_notebook(path)
    paths = [nb_dir, root]
    src = root / "src"
    if src.exists(): paths.append(src)
    return [p for i, p in enumerate(paths) if p.exists() and p not in paths[:i]]

In [ ]:
#| export
def _exec_shell(path, extra_paths=None):
    shell = CaptureShell()
    for pth in reversed([*(_local_import_paths(path)), *(extra_paths or [])]):
        shell.set_path(pth)
    return shell

### Timeouts that do not trap future runs

A timed-out cell can make repeated verification painful. Timeout metadata records the source hash that timed out, skips that exact source on later runs, and automatically clears the mark when the cell changes.

In [ ]:
#| export
_TIMEOUT_HASH_KEY = "nbskill_timeout_hash"

In [ ]:
#| export
_TIMEOUT_SECONDS_KEY = "nbskill_timeout_seconds"

In [ ]:
#| export
def _source_hash(source):
    return hashlib.sha256(str(source).encode("utf-8")).hexdigest()

In [ ]:
#| export
# _cell_metadata is imported from nbskill.foundation.

In [ ]:
#| export
def _cell_source_hash(cell): return _source_hash(cell.get("source", ""))

In [ ]:
#| export
def _timeout_stream(text):
    return {"output_type": "stream", "name": "stderr", "text": text if text.endswith("\n") else text + "\n"}

In [ ]:
#| export
def _timeout_error(ename, text):
    return {"output_type": "error", "ename": ename, "evalue": text, "traceback": [text]}

In [ ]:
#| export
def _skip_timed_out_cell(cell):
    if cell.cell_type != "code": return False
    meta = cell_metadata(cell)
    current_hash = _cell_source_hash(cell)
    timeout_hash = meta.get(_TIMEOUT_HASH_KEY)
    if timeout_hash == current_hash:
        seconds = meta.get(_TIMEOUT_SECONDS_KEY, "?")
        msg = (
            f"nbskill: skipped cell id={cell.id}; it previously exceeded the "
            f"{seconds}s timeout. Edit the cell to change its hash and rerun it."
        )
        cell.outputs = [_timeout_error("NbskillTimeoutSkipped", msg)]
        return True
    if timeout_hash and timeout_hash != current_hash:
        meta.pop(_TIMEOUT_HASH_KEY, None)
        meta.pop(_TIMEOUT_SECONDS_KEY, None)
    return False

In [ ]:
#| export
def _mark_timeout(cell, timeout, outputs):
    meta = cell_metadata(cell)
    meta[_TIMEOUT_HASH_KEY] = _cell_source_hash(cell)
    meta[_TIMEOUT_SECONDS_KEY] = timeout
    msg = f"nbskill: cell id={cell.id} ran longer than {timeout}s and was stopped."
    cell.outputs = [_timeout_stream(msg), *(outputs or [])]

In [ ]:
#| export
def _clear_timeout_mark(cell):
    meta = cell_metadata(cell)
    meta.pop(_TIMEOUT_HASH_KEY, None)
    meta.pop(_TIMEOUT_SECONDS_KEY, None)

In [ ]:
#| export
def _run_cell(shell, cell, timeout=30, verbose=False):
    if cell.cell_type != "code": return
    shell._cell_idx = cell.idx_ + 1
    outputs = shell.run(cell.source, timeout=timeout if timeout and timeout > 0 else None, verbose=verbose)
    cell.outputs = outputs or []
    if isinstance(shell.exc, TimeoutError): _mark_timeout(cell, timeout, outputs)
    else: _clear_timeout_mark(cell)

In [ ]:
#| export
def _execute_nb(path, dest=None, exc_stop=False, preproc=lambda cell: False, postproc=lambda cell: None, timeout=30, verbose=False):
    with notebook_locks(path, dest):
        with execution_slot():
            nb = _read_nb(path)
            shell = _exec_shell(path)
            first_exc = None
            for cell in nb.cells:
                if preproc(cell): continue
                if _skip_timed_out_cell(cell):
                    postproc(cell)
                    continue
                _run_cell(shell, cell, timeout=timeout, verbose=verbose)
                postproc(cell)
                if shell.exc and exc_stop:
                    first_exc = shell.exc
                    break
            if dest:
                stamp_notebook_metadata(nb)
                _write_nb(nb, dest)
            if first_exc: raise first_exc
            return nb

In [ ]:
#| export
def _text_output(value):
    if value is None: return ""
    if isinstance(value, list): return "".join(map(str, value))
    return str(value)


def _output_text(output):
    otype = output.get("output_type")
    if otype == "stream": return _text_output(output.get("text"))
    if otype == "error":
        tb = output.get("traceback")
        if tb: return _text_output(tb)
        return f"{output.get('ename', 'Error')}: {output.get('evalue', '')}"
    if otype in {"execute_result", "display_data"}:
        data = output.get("data", {})
        for mime in ("text/plain", "text/markdown", "text/html"):
            if mime in data: return _text_output(data[mime])
    return ""


def _is_rich_traceback_stream(output):
    if output.get("output_type") != "stream": return False
    text = _text_output(output.get("text"))
    return "Traceback" in text and "\x1b[" in text


def _executed_cells(nb, up2id=None):
    up2id = parse_literal(up2id)
    if up2id is None: return list(enumerate(nb.cells))
    if isinstance(up2id, int): return list(enumerate(nb.cells[:up2id]))
    items = []
    for idx, cell in enumerate(nb.cells):
        items.append((idx, cell))
        if cell.id == str(up2id): break
    return items


def _print_nb_outputs(path, up2id=None):
    with notebook_locks(path):
        nb = _read_nb(path)
        for idx, cell in _executed_cells(nb, up2id):
            outputs = getattr(cell, "outputs", None) or []
            has_error = any(output.get("output_type") == "error" for output in outputs)
            for output in outputs:
                if has_error and _is_rich_traceback_stream(output): continue
                text = _output_text(output)
                if not text: continue
                print(f"--- output id={cell.id} ---")
                print(text, end="" if text.endswith("\n") else "\n")

### The public executor

`exec_nb` is the user-facing wrapper around the execution engine. It writes outputs back to the notebook, prints visible outputs when requested, and supports partial execution through `up2id` or `chapter`.

In [ ]:
#| export
@call_parse
@tracked_call
def exec_nb(
    path: str,  # Notebook path
    dest: str | None = None,  # Destination path; defaults to overwriting path
    exc_stop: bool = False,  # Stop on exceptions
    up2id: int | str | None = None,  # Execute first N cells, or through this cell id
    chapter: str | None = None,  # Execute through this chapter, inclusive
    timeout: int = 30,  # Per-cell timeout in seconds; <=0 disables timeouts
    show_output: bool = True,  # Print saved cell outputs and errors after execution
    verbose: bool = False,  # Show stdout/stderr live while executing
):
    "Execute a notebook with execnb and local project imports available."
    dest = dest or path
    chapter_title = None
    if chapter is not None:
        if up2id is not None: raise ValueError("Use either chapter or up2id, not both")
        with notebook_locks(path):
            nb = _read_nb(path)
            span = one_chapter(nb.cells, chapter)
        up2id, chapter_title = span["end"], span["title"]
    preproc, postproc = _exec_limiters(up2id)
    _execute_nb(path, dest=dest, exc_stop=exc_stop, preproc=preproc, postproc=postproc, timeout=timeout, verbose=verbose)
    msg = f"Executed {path} -> {dest}"
    if chapter_title is not None: msg += f" (chapter={chapter_title!r}, up2id={up2id})"
    elif up2id is not None: msg += f" (up2id={up2id})"
    if timeout and timeout > 0: msg += f" (timeout={timeout}s)"
    print(msg)
    if show_output: _print_nb_outputs(dest, up2id=up2id)
    return cli_return(Path(dest))

In [ ]:
import tempfile as _tempfile
from pathlib import Path as _Path
from fastcore.nbio import read_nb as _read_nb
from nbskill.execute import _TIMEOUT_HASH_KEY, _output_text, exec_nb
from nbskill.write import update_cell, write_nb

with _tempfile.TemporaryDirectory() as td:
    root = _Path(td)
    (root / "pyproject.toml").write_text("[project]\nname = 'local-demo'\n", encoding="utf-8")
    pkg = root / "local_demo"
    pkg.mkdir()
    (pkg / "__init__.py").write_text("def meaning():\n    return 42\n", encoding="utf-8")
    nbs = root / "nbs"
    nbs.mkdir()
    path = nbs / "sample.ipynb"
    write_nb(str(path), "%%code\nfrom local_demo import meaning\nprint(meaning())\nassert meaning() == 42", replace=True, export=False)
    exec_nb(str(path), timeout=5)
    text = "".join(_output_text(output) for output in _read_nb(path).cells[0].outputs)
    assert "42" in text

    slow = nbs / "slow.ipynb"
    write_nb(str(slow), "%%code\nimport time\ntime.sleep(2)", replace=True, export=False)
    exec_nb(str(slow), timeout=1)
    cell = _read_nb(slow).cells[0]
    assert cell.metadata[_TIMEOUT_HASH_KEY]
    text = "".join(_output_text(output) for output in cell.outputs)
    assert "ran longer than 1s" in text
    exec_nb(str(slow), timeout=1)
    cell = _read_nb(slow).cells[0]
    text = "".join(_output_text(output) for output in cell.outputs)
    assert "skipped cell" in text
    update_cell(str(slow), "print('fast now')", cell_id=cell.id, export=False)
    exec_nb(str(slow), timeout=1)
    cell = _read_nb(slow).cells[0]
    assert _TIMEOUT_HASH_KEY not in cell.metadata
    text = "".join(_output_text(output) for output in cell.outputs)
    assert "fast now" in text

### Testing after edits

The write tools can ask for a notebook test immediately after changing a notebook. These helpers summarize saved error outputs so a failed write/test cycle reports the useful cell-level problem.

In [ ]:
#| export
def _notebook_error_summaries(path, up2id=None):
    with notebook_locks(path):
        nb = _read_nb(path)
        errors = []
        for idx, cell in _executed_cells(nb, up2id=up2id):
            for output in cell.get("outputs", []):
                if output.get("output_type") == "error":
                    ename = output.get("ename", "Error")
                    evalue = output.get("evalue", "")
                    errors.append(f"id={cell.id} {ename}: {evalue}".strip())
        return errors

In [ ]:
#| export
def run_notebook_test(path, timeout=30):
    print(f"Running notebook test with execnb on {path} (timeout={timeout}s)")
    _execute_nb(path, dest=path, exc_stop=False, timeout=timeout, verbose=False)
    _print_nb_outputs(path)
    errors = _notebook_error_summaries(path)
    if errors:
        sys.stdout.flush()
        cli_error("Notebook test failed after writing/execution: " + "; ".join(errors))